# Position Backtest Validation

Notebook de QA visuelle et comptable pour les backtests de positions simulées.

Couverture:
- inventaire des scenarios materialises
- checks comptables sur `mart_position_backtest_summary`
- inspection swap-level de `int_position_backtest_swap_path`
- visus start/end composition, fees cumulees, prix et statut in-range

Objectif:
- verifier a la main que les chiffres craches par le pipeline sont coherents avant d'aller plus loin


## Prerequis

- `clickhouse_local` doit tourner sur `localhost:8124`
- la slice backtest doit etre build avec au moins un scenario

Exemple de build local:

```bash
cd /home/gharbi/defind-analytics/dbt/defind_analytics
../../.venv/bin/dbt build --target local \
  --select int_swaps_enriched int_position_backtest_inputs int_position_backtest_entry_state int_position_backtest_entry_liquidity int_position_backtest_swap_path int_position_backtest_time_in_range int_position_backtest_exit_state mart_position_backtest_summary \
  --vars '{"position_backtest_scenarios":[{"scenario_id":"demo_1","pool":"0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640","start_block":24810000,"end_block":24820333,"tick_lower":198000,"tick_upper":201000,"capital_committed_token0":100.0}]}'
```

Note:
- les metriques `observed_*` de temps sont bien des metriques observees sur la timeline des swaps, pas encore une verite exacte block-par-block sur les heures silencieuses.


In [13]:
import json
import os
import subprocess
from decimal import Decimal
from pathlib import Path
from textwrap import dedent

import clickhouse_connect

try:
    from IPython.display import display, Markdown
except ModuleNotFoundError:
    def display(obj):
        print(obj)
    def Markdown(text):
        return text

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "pandas is required. Run: %pip install pandas clickhouse-connect plotly jupyterlab"
    ) from exc

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "plotly is required. Run: %pip install plotly"
    ) from exc


In [14]:
REPO_ROOT = Path(os.getenv("DEFIND_REPO_ROOT", "/home/gharbi/defind-analytics"))
DBT_PROJECT_DIR = REPO_ROOT / "dbt/defind_analytics"
DBT_BIN = REPO_ROOT / ".venv/bin/dbt"
DBT_BUILD_SELECTION = [
    "int_swaps_enriched",
    "int_position_backtest_inputs",
    "int_position_backtest_entry_state",
    "int_position_backtest_entry_liquidity",
    "int_position_backtest_swap_path",
    "int_position_backtest_time_in_range",
    "int_position_backtest_exit_state",
    "mart_position_backtest_summary",
    "test_mart_position_backtest_summary_unique_scenario",
]

CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", "localhost")
CLICKHOUSE_PORT = int(os.getenv("CLICKHOUSE_PORT", "8124"))
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "default")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "1234")
CLICKHOUSE_DATABASE = os.getenv("CLICKHOUSE_DATABASE", "analytics")
BACKTEST_SCENARIO_ID = os.getenv("BACKTEST_SCENARIO_ID")

client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

TIME_COLUMNS = [
    "date",
    "hour",
    "week_start",
    "block_time",
    "start_snapshot_block_time",
    "end_snapshot_block_time",
    "resolved_start_time",
    "resolved_end_time",
]

def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    for column in TIME_COLUMNS:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], errors="coerce")
    for column in df.columns:
        if column in TIME_COLUMNS:
            continue
        if df[column].dtype == "object":
            non_null = next((value for value in df[column] if value is not None and not pd.isna(value)), None)
            if isinstance(non_null, Decimal):
                df[column] = df[column].map(lambda value: float(value) if value is not None and not pd.isna(value) else None)
    return df

def query_df(sql: str, parameters: dict | None = None) -> pd.DataFrame:
    return normalize_df(client.query_df(sql, parameters=parameters or {}))

def run_backtest_build(scenario_config: dict, target: str = "local") -> None:
    vars_payload = {"position_backtest_scenarios": [scenario_config]}
    cmd = [
        str(DBT_BIN),
        "build",
        "--target",
        target,
        "--select",
        *DBT_BUILD_SELECTION,
        "--vars",
        json.dumps(vars_payload),
    ]
    print("Running:")
    print(" ".join(cmd))
    result = subprocess.run(
        cmd,
        cwd=DBT_PROJECT_DIR,
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        tail = "\n".join((result.stdout + "\n" + result.stderr).splitlines()[-80:])
        raise RuntimeError(f"dbt build failed for scenario {scenario_config['scenario_id']}\n\n{tail}")
    print("dbt build completed successfully.")
    for line in result.stdout.splitlines()[-20:]:
        print(line)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


In [ ]:
# Optional: build a scenario directly from the notebook.
# Leave RUN_DBT_BUILD = False if you only want to inspect already-materialized scenarios.
# Important: if RUN_DBT_BUILD stays False, changing start/end blocks or ticks here does nothing
# until you rebuild. In that mode, the notebook only reads whatever scenario_id already exists in ClickHouse.

RUN_DBT_BUILD = True
BACKTEST_SCENARIO_CONFIG = {
    "scenario_id": os.getenv("BACKTEST_SCENARIO_ID", "demo_recent_midrange"),
    "pool": os.getenv("BACKTEST_POOL", "0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640"),
    "start_block": int(os.getenv("BACKTEST_START_BLOCK", "24790000")),
    "end_block": int(os.getenv("BACKTEST_END_BLOCK", "24820333")),
    "tick_lower": int(os.getenv("BACKTEST_TICK_LOWER", "198000")),
    "tick_upper": int(os.getenv("BACKTEST_TICK_UPPER", "201000")),
    "capital_committed_token0": float(os.getenv("BACKTEST_CAPITAL_TOKEN0", "100.0")),
}

display(pd.DataFrame([BACKTEST_SCENARIO_CONFIG]))

if RUN_DBT_BUILD:
    run_backtest_build(BACKTEST_SCENARIO_CONFIG)
    BACKTEST_SCENARIO_ID = BACKTEST_SCENARIO_CONFIG["scenario_id"]


,scenario_id,pool,start_block,end_block,tick_lower,tick_upper,capital_committed_token0
0,demo_recent_midrange,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,24800000,24820333,198000,201000,100.0


Running:
/home/gharbi/defind-analytics/.venv/bin/dbt build --target local --select int_swaps_enriched int_position_backtest_inputs int_position_backtest_entry_state int_position_backtest_entry_liquidity int_position_backtest_swap_path int_position_backtest_time_in_range int_position_backtest_exit_state mart_position_backtest_summary test_mart_position_backtest_summary_unique_scenario --vars {"position_backtest_scenarios": [{"scenario_id": "demo_recent_midrange", "pool": "0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640", "start_block": 24800000, "end_block": 24820333, "tick_lower": 198000, "tick_upper": 201000, "capital_committed_token0": 100.0}]}
dbt build completed successfully.
16:17:53  43 of 49 START test not_null_mart_position_backtest_summary_end_block .......... [RUN]
16:17:53  43 of 49 PASS not_null_mart_position_backtest_summary_end_block ................ [PASS in 0.05s]
16:17:54  44 of 49 START test not_null_mart_position_backtest_summary_pool ............... [RUN]
16:17:54  44 of

In [16]:
available_pools_df = query_df("SELECT pool FROM stg_pools ORDER BY pool")
scenario_inventory_df = query_df(
    dedent(
        """
        SELECT
            scenario_id,
            pool,
            start_block,
            end_block,
            tick_lower,
            tick_upper,
            capital_committed_token0,
            start_snapshot_block_time,
            end_snapshot_block_time,
            swap_count_in_window,
            swap_count_while_active,
            observed_time_in_range_pct,
            pnl_vs_initial_pct,
            pnl_vs_hodl_pct
        FROM mart_position_backtest_summary
        ORDER BY start_block, end_block, scenario_id
        """
    )
)
display(scenario_inventory_df)

if scenario_inventory_df.empty:
    default_pool = available_pools_df["pool"].iloc[0] if not available_pools_df.empty else "0xPOOL"
    example_cmd = dedent(
        f"""
        No backtest scenario is currently materialized.

        Example build command:

        cd /home/gharbi/defind-analytics/dbt/defind_analytics
        ../../.venv/bin/dbt build --target local \\
          --select int_swaps_enriched int_position_backtest_inputs int_position_backtest_entry_state int_position_backtest_entry_liquidity int_position_backtest_swap_path int_position_backtest_time_in_range int_position_backtest_exit_state mart_position_backtest_summary \\
          --vars '{{"position_backtest_scenarios":[{{"scenario_id":"demo_1","pool":"{default_pool}","start_block":24810000,"end_block":24820333,"tick_lower":198000,"tick_upper":201000,"capital_committed_token0":100.0}}]}}'
        """
    )
    display(Markdown(f"```text\n{example_cmd}\n```"))
    raise RuntimeError("No materialized backtest scenario found. Build one first, then rerun the notebook.")


,scenario_id,pool,start_block,end_block,tick_lower,tick_upper,capital_committed_token0,start_snapshot_block_time,end_snapshot_block_time,swap_count_in_window,swap_count_while_active,observed_time_in_range_pct,pnl_vs_initial_pct,pnl_vs_hodl_pct
0,demo_recent_midrange,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,24800000,24820333,198000,201000,100.0,2026-04-03 15:27:47,2026-04-06 11:28:11,8310,8310,1.0,0.029478,-0.002445


In [17]:
SCENARIO_ID = BACKTEST_SCENARIO_CONFIG.get("scenario_id") or BACKTEST_SCENARIO_ID or scenario_inventory_df["scenario_id"].iloc[0]
print(f"Using scenario_id={SCENARIO_ID}")

summary_df = query_df(
    "SELECT * FROM mart_position_backtest_summary WHERE scenario_id = %(scenario_id)s",
    {"scenario_id": SCENARIO_ID},
)
if summary_df.empty:
    raise RuntimeError(
        f"Scenario {SCENARIO_ID} was not found in mart_position_backtest_summary. "
        "If you changed BACKTEST_SCENARIO_CONFIG, set RUN_DBT_BUILD = True and rerun from the config cell."
    )

summary_row = summary_df.iloc[0].copy()
display(summary_df.T)


Using scenario_id=demo_recent_midrange


,0
scenario_id,demo_recent_midrange
pool,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640
start_block,24800000
end_block,24820333
tick_lower,198000
tick_upper,201000
capital_committed_token0,100.0
rebalance_policy,none
start_snapshot_block_number,24800000
start_snapshot_log_index,373


In [18]:
accounting_checks_df = pd.DataFrame(
    [
        {
            "check": "initial_principal_equals_committed_capital",
            "lhs": float(summary_row["initial_principal_value_token0"]),
            "rhs": float(summary_row["capital_committed_token0"]),
        },
        {
            "check": "hodl_end_value_rebuild",
            "lhs": float(summary_row["hodl_end_value_token0"]),
            "rhs": float(summary_row["initial_amount0"] + summary_row["initial_amount1"] * summary_row["end_price"]),
        },
        {
            "check": "end_principal_value_rebuild",
            "lhs": float(summary_row["end_principal_value_token0"]),
            "rhs": float(summary_row["end_amount0"] + summary_row["end_amount1"] * summary_row["end_price"]),
        },
        {
            "check": "lp_end_including_fees_rebuild",
            "lhs": float(summary_row["lp_end_value_including_fees_token0"]),
            "rhs": float(summary_row["end_principal_value_token0"] + summary_row["fees_accrued_token0_equiv_generated"]),
        },
        {
            "check": "impermanent_loss_rebuild",
            "lhs": float(summary_row["impermanent_loss_token0"]),
            "rhs": float(summary_row["end_principal_value_token0"] - summary_row["hodl_end_value_token0"]),
        },
        {
            "check": "pnl_vs_initial_rebuild",
            "lhs": float(summary_row["pnl_vs_initial_token0"]),
            "rhs": float(summary_row["lp_end_value_including_fees_token0"] - summary_row["capital_committed_token0"]),
        },
        {
            "check": "pnl_vs_hodl_rebuild",
            "lhs": float(summary_row["pnl_vs_hodl_token0"]),
            "rhs": float(summary_row["lp_end_value_including_fees_token0"] - summary_row["hodl_end_value_token0"]),
        },
        {
            "check": "observed_time_reconciliation",
            "lhs": float(summary_row["observed_window_seconds"]),
            "rhs": float(summary_row["observed_time_in_range_seconds"] + summary_row["observed_out_of_range_seconds"]),
        },
    ]
)
accounting_checks_df["gap"] = accounting_checks_df["lhs"] - accounting_checks_df["rhs"]
accounting_checks_df["abs_gap"] = accounting_checks_df["gap"].abs()
display(accounting_checks_df)


,check,lhs,rhs,gap,abs_gap
0,initial_principal_equals_committed_capital,100.000000,100.000000,0.0,0.0
1,hodl_end_value_rebuild,103.200198,103.200198,0.0,0.0
2,end_principal_value_rebuild,102.820883,102.820883,0.0,0.0
3,lp_end_including_fees_rebuild,102.947824,102.947824,0.0,0.0
4,impermanent_loss_rebuild,-0.379315,-0.379315,0.0,0.0
5,pnl_vs_initial_rebuild,2.947824,2.947824,0.0,0.0
6,pnl_vs_hodl_rebuild,-0.252374,-0.252374,0.0,0.0
7,observed_time_reconciliation,244824.000000,244824.000000,0.0,0.0


In [19]:
path_df = query_df(
    dedent(
        """
        SELECT
            p.scenario_id,
            p.pool,
            p.block_number,
            p.log_index,
            p.block_time,
            p.tick_before,
            p.price_before,
            p.tick_after,
            p.price_after,
            p.active_liquidity_at_swap,
            p.initial_liquidity,
            p.is_active_before_swap,
            p.position_fee_share,
            p.fees_generated_token0,
            p.fees_generated_token1,
            p.fees_generated_token0_equiv,
            p.fees_accrued_token0,
            p.fees_accrued_token1,
            p.fees_accrued_token0_equiv_generated,
            s.tick_lower,
            s.tick_upper,
            s.start_price,
            s.end_price
        FROM int_position_backtest_swap_path p
        INNER JOIN mart_position_backtest_summary s USING (scenario_id)
        WHERE p.scenario_id = %(scenario_id)s
        ORDER BY p.block_number, p.log_index
        """
    ),
    {"scenario_id": SCENARIO_ID},
)
path_df["fees_accrued_token0_equiv_cum"] = path_df["fees_accrued_token0_equiv_generated"].fillna(0.0).cumsum()
path_df["active_label"] = path_df["is_active_before_swap"].map({1: "in_range", 0: "out_of_range"})
display(path_df.head(20))


,scenario_id,pool,block_number,log_index,block_time,tick_before,price_before,tick_after,price_after,active_liquidity_at_swap,initial_liquidity,is_active_before_swap,position_fee_share,fees_generated_token0,fees_generated_token1,fees_generated_token0_equiv,fees_accrued_token0,fees_accrued_token1,fees_accrued_token0_equiv_generated,s.tick_lower,s.tick_upper,start_price,end_price,fees_accrued_token0_equiv_cum,active_label
0,demo_recent_midrange,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,24800004,143,2026-04-03 15:28:35,200049,2053.071098,200049,2053.071957,1.380108e+19,1.534688e+13,1,0.000001,0.065415,0.000000,0.065415,7.274177e-08,0.000000e+00,7.274177e-08,198000,201000,2053.071098,2150.092004,7.274177e-08,in_range
1,demo_recent_midrange,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,24800005,234,2026-04-03 15:28:47,200049,2053.071957,200049,2053.072315,1.380108e+19,1.534688e+13,1,0.000001,0.027279,0.000000,0.027279,3.033458e-08,0.000000e+00,3.033458e-08,198000,201000,2053.071098,2150.092004,1.030763e-07,in_range
2,demo_recent_midrange,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,24800005,431,2026-04-03 15:28:47,200049,2053.072315,200049,2053.072317,1.380108e+19,1.534688e+13,1,0.000001,0.000209,0.000000,0.000209,2.326361e-10,0.000000e+00,2.326361e-10,198000,201000,2053.071098,2150.092004,1.033090e-07,in_range
3,demo_recent_midrange,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,24800006,281,2026-04-03 15:28:59,200049,2053.072317,200049,2053.072424,1.380108e+19,1.534688e+13,1,0.000001,0.008107,0.000000,0.008107,9.015108e-09,0.000000e+00,9.015108e-09,198000,201000,2053.071098,2150.092004,1.123241e-07,in_range
4,demo_recent_midrange,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,24800006,570,2026-04-03 15:28:59,200049,2053.072424,200049,2053.072484,1.380108e+19,1.534688e+13,1,0.000001,0.004592,0.000000,0.004592,5.106881e-09,0.000000e+00,5.106881e-09,198000,201000,2053.071098,2150.092004,1.174310e-07,in_range
5,demo_recent_midrange,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,24800013,144,2026-04-03 15:30:23,200049,2053.072484,200049,2053.072920,1.380108e+19,1.534688e+13,1,0.000001,0.033187,0.000000,0.033187,3.690402e-08,0.000000e+00,3.690402e-08,198000,201000,2053.071098,2150.092004,1.543350e-07,in_range
6,demo_recent_midrange,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,24800013,168,2026-04-03 15:30:23,200049,2053.072920,200049,2053.072987,1.380108e+19,1.534688e+13,1,0.000001,0.005114,0.000000,0.005114,5.686973e-09,0.000000e+00,5.686973e-09,198000,201000,2053.071098,2150.092004,1.600220e-07,in_range
7,demo_recent_midrange,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,24800013,381,2026-04-03 15:30:23,200049,2053.072987,200049,2053.073358,1.380108e+19,1.534688e+13,1,0.000001,0.028309,0.000000,0.028309,3.147998e-08,0.000000e+00,3.147998e-08,198000,201000,2053.071098,2150.092004,1.915019e-07,in_range
8,demo_recent_midrange,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,24800018,54,2026-04-03 15:31:23,200049,2053.073358,200049,2053.073080,1.380174e+19,1.534688e+13,1,0.000001,0.000000,0.000010,0.021242,0.000000e+00,1.150489e-11,2.362037e-08,198000,201000,2053.071098,2150.092004,2.151223e-07,in_range
9,demo_recent_midrange,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,24800019,184,2026-04-03 15:31:35,200049,2053.073080,200049,2053.110872,1.380174e+19,1.534688e+13,1,0.000001,2.879348,0.000000,2.879348,3.201699e-06,0.000000e+00,3.201699e-06,198000,201000,2053.071098,2150.092004,3.416822e-06,in_range


In [20]:
path_checks_df = pd.DataFrame(
    [
        {
            "metric": "swap_rows_in_path",
            "value": int(len(path_df)),
        },
        {
            "metric": "active_rows_in_path",
            "value": int(path_df["is_active_before_swap"].fillna(0).sum()),
        },
        {
            "metric": "summary_swap_count_in_window",
            "value": int(summary_row["swap_count_in_window"]),
        },
        {
            "metric": "summary_swap_count_while_active",
            "value": int(summary_row["swap_count_while_active"]),
        },
        {
            "metric": "path_fee_sum_token0_equiv",
            "value": float(path_df["fees_accrued_token0_equiv_generated"].fillna(0.0).sum()),
        },
        {
            "metric": "summary_fee_sum_token0_equiv",
            "value": float(summary_row["fees_accrued_token0_equiv_generated"]),
        },
        {
            "metric": "max_position_fee_share",
            "value": float(path_df["position_fee_share"].fillna(0.0).max()) if not path_df.empty else 0.0,
        },
        {
            "metric": "inactive_rows_with_nonzero_fee",
            "value": int(((path_df["is_active_before_swap"] == 0) & (path_df["fees_accrued_token0_equiv_generated"].fillna(0.0) > 0)).sum()),
        },
    ]
)
display(path_checks_df)


,metric,value
0,swap_rows_in_path,8310.000000
1,active_rows_in_path,8310.000000
2,summary_swap_count_in_window,8310.000000
3,summary_swap_count_while_active,8310.000000
4,path_fee_sum_token0_equiv,0.126942
5,summary_fee_sum_token0_equiv,0.126942
6,max_position_fee_share,0.000001
7,inactive_rows_with_nonzero_fee,0.000000


In [21]:
value_compare_df = pd.DataFrame(
    {
        "series": [
            "initial_capital",
            "hodl_end_value",
            "lp_principal_end_value",
            "lp_end_value_including_fees",
            "fees_accrued_token0_equiv",
        ],
        "value": [
            float(summary_row["capital_committed_token0"]),
            float(summary_row["hodl_end_value_token0"]),
            float(summary_row["end_principal_value_token0"]),
            float(summary_row["lp_end_value_including_fees_token0"]),
            float(summary_row["fees_accrued_token0_equiv_generated"]),
        ],
    }
)

fig = px.bar(
    value_compare_df,
    x="series",
    y="value",
    title=f"Scenario value decomposition - {SCENARIO_ID}",
    template="plotly_white",
    text_auto=".4f",
)
fig.update_layout(xaxis_title="", yaxis_title="token0 value")
fig.show()


In [22]:
composition_df = pd.DataFrame(
    [
        {
            "snapshot": "start",
            "token0_component": float(summary_row["initial_amount0"]),
            "token1_component_in_token0": float(summary_row["initial_amount1"] * summary_row["start_price"]),
        },
        {
            "snapshot": "end_principal",
            "token0_component": float(summary_row["end_amount0"]),
            "token1_component_in_token0": float(summary_row["end_amount1"] * summary_row["end_price"]),
        },
    ]
)
composition_long_df = composition_df.melt(id_vars="snapshot", var_name="component", value_name="value")
fig = px.bar(
    composition_long_df,
    x="snapshot",
    y="value",
    color="component",
    barmode="stack",
    title=f"Start vs end principal composition - {SCENARIO_ID}",
    template="plotly_white",
)
fig.update_layout(yaxis_title="token0 value")
fig.show()


In [23]:
if path_df.empty:
    display(Markdown("No swap path rows were materialized for this scenario."))
else:
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Scatter(
            x=path_df["block_time"],
            y=path_df["price_before"],
            mode="lines+markers",
            name="price_before",
            marker=dict(color=path_df["is_active_before_swap"], colorscale="RdYlGn", size=7, showscale=False),
            text=path_df["active_label"],
            hovertemplate="%{x}<br>price=%{y}<br>%{text}<extra></extra>",
        ),
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=path_df["block_time"],
            y=path_df["fees_accrued_token0_equiv_cum"],
            mode="lines+markers",
            name="cumulative_fees_token0_equiv",
            line=dict(color="#d62728"),
        ),
        secondary_y=True,
    )
    fig.update_layout(
        title=f"Price path and cumulative accrued fees - {SCENARIO_ID}",
        template="plotly_white",
        hovermode="x unified",
    )
    fig.update_yaxes(title_text="price_token1_in_token0", secondary_y=False)
    fig.update_yaxes(title_text="cumulative fees in token0", secondary_y=True)
    fig.show()


In [24]:
if path_df.empty:
    display(Markdown("No swap path rows were materialized for this scenario."))
else:
    status_df = path_df.copy()
    status_df["active_numeric"] = status_df["is_active_before_swap"].fillna(0).astype(int)
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Scatter(
            x=status_df["block_time"],
            y=status_df["active_numeric"],
            name="is_active_before_swap",
            mode="lines",
            line=dict(color="#2ca02c", shape="hv"),
        ),
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=status_df["block_time"],
            y=status_df["tick_before"],
            mode="lines+markers",
            name="tick_before",
            line=dict(color="#1f77b4"),
        ),
        secondary_y=True,
    )
    fig.add_hline(y=summary_row["tick_lower"], line_dash="dot", line_color="#999999", secondary_y=True)
    fig.add_hline(y=summary_row["tick_upper"], line_dash="dot", line_color="#999999", secondary_y=True)
    fig.update_layout(
        title=f"In-range status and tick path - {SCENARIO_ID}",
        template="plotly_white",
        hovermode="x unified",
    )
    fig.update_yaxes(title_text="active flag", range=[-0.1, 1.1], secondary_y=False)
    fig.update_yaxes(title_text="tick_before", secondary_y=True)
    fig.show()


AttributeError: module 'plotly.graph_objects' has no attribute 'Step'